# CSC 2301 · Week 14 — Final project: evaluating well KW-3

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W14_final_project_kw3.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Well KW-3 has just reached total depth. The logging company sent the raw file `well_log_kw3.csv` (synthetic): gamma ray,
bulk density and deep resistivity every 0.5 m from 1800 to 2060 m. Some samples are missing (the tool wrote
the null code `-999.25`). Your team must tell the asset manager, **in one notebook and a 5-minute
presentation**: where the pay is, how thick it is, and how much oil it may hold.

Every part reuses the course: reading and cleaning (W4, W6, W10), functions (W3, W11), masks (W8),
runs of intervals (W7), figures (W9), Monte Carlo (W13).

**Deliverables:** this notebook with all checks passing, `kw3_logs.png`, `kw3_summary.txt`, and 4 slides
(see the end of the notebook).

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/well_log_kw3.csv', 'well_log_kw3.csv'), ('tests/w14/q1.py', 'tests/q1.py'), ('tests/w14/q2.py', 'tests/q2.py'), ('tests/w14/q3.py', 'tests/q3.py'), ('tests/w14/q4.py', 'tests/q4.py'), ('tests/w14/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## The rules of the evaluation

| quantity | formula | parameters |
|---|---|---|
| shale volume | $V_{sh} = (GR - 20) / (140 - 20)$, clipped to 0–1 | clean line 20 API, shale line 140 API |
| porosity | $\phi = (2.65 - \rho_b) / (2.65 - 1.0)$, clipped to 0–0.40 | sandstone matrix, water |
| **pay** | $V_{sh} < 0.35$ **and** $\phi \ge 0.12$ **and** $R_t \ge 15$ Ω·m | a missing value is never pay |
| pay zone | consecutive pay samples, at least 1.0 m thick | each sample = 0.5 m |

A sand with **low resistivity** is porous but full of salt water: it is not pay. Look for it in the figure.

Start by looking at the file.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("well_log_kw3.csv")
print(raw.shape)
raw.head()

In [ ]:
raw.describe()     # what is strange in the min row?

---
## Project tasks

### Q1. Load and clean

Read the file into `log` and replace the null code `-999.25` by `np.nan` (keep **every** row: the depths must
stay regular). Store in `n_bad` the number of depths with at least one missing value.

In [ ]:
log = ...
n_bad = ...
n_bad, len(log)

In [ ]:
grader.check("q1")

### Q2. A petrophysics function

Write `evaluate(df)` that returns a **copy** of the table with three new columns: `vsh`, `phi` and `pay`
(Boolean), following the rules above. No loop: column operations, `.clip`, `&`. Then `kw3 = evaluate(log)`.

*Note:* a comparison with `NaN` is `False`, so a sample with a missing value is automatically not pay.

In [ ]:
def evaluate(df):
    """Add vsh, phi and pay columns to a copy of a log table."""
    out = df.copy()
    ...
    return out

kw3 = evaluate(log)
kw3.head()

In [ ]:
grader.check("q2")

### Q3. Pay zones

Write `pay_zones(depths, pay, min_m=1.0)` that returns the list of zones `(top_m, base_m)`: runs of consecutive
pay samples, with `top` = first depth and `base` = last depth + 0.5 m, keeping only zones at least `min_m` thick.
(It is the week 7 intercept algorithm again.) Then store:

* `zones = pay_zones(kw3["depth_m"], kw3["pay"])`
* `net_pay_m`: total thickness of the zones
* `phi_pay`: mean porosity of the pay samples

Look at your zones: are two of them separated by **one missing sample**? Say in your presentation whether
you would report them as one zone, and why.

In [ ]:
def pay_zones(depths, pay, min_m=1.0):
    ...

zones = ...
net_pay_m = ...
phi_pay = ...
zones, net_pay_m, phi_pay

In [ ]:
grader.check("q3")

### Q4. The log display

Build `fig_kw3` with **three tracks** sharing depth: GR, RHOB and RT. Depth downwards, every axis labelled with
its unit, **RT on a logarithmic scale**, and each pay zone shaded in the GR track with
`ax.axhspan(top, base, color="gold", alpha=0.3)` (you may shade all tracks). Give it a title and save it as
`kw3_logs.png` (150 dpi, `bbox_inches="tight"`).

In [ ]:
fig_kw3, (ax1, ax2, ax3) = plt.subplots(1, 3, sharey=True, figsize=(9, 10))
...
plt.show()

In [ ]:
grader.check("q4")

### Q5. How much oil? With uncertainty

Run a Monte Carlo of 100 000 draws with `rng = np.random.default_rng(14)`:

| parameter | distribution |
|---|---|
| area (km²) | triangular: min 2, most likely 4, max 7 |
| thickness h (m) | fixed: `net_pay_m` |
| porosity | normal(`phi_pay`, 0.02), clipped to 0.05–0.35 |
| Sw | uniform(0.25, 0.45) |
| Bo | fixed: 1.2 |

Store the results in `stoiip` (MMbbl, your week 13 formula) and `p90, p50, p10`. Then write
`kw3_summary.txt`, one line exactly in this format (your numbers):

```
KW-3: 4 pay zones, net pay 40.5 m, mean porosity 0.218, STOIIP P90 51.2 / P50 76.4 / P10 107.3 MMbbl
```

In [ ]:
rng = np.random.default_rng(14)
N = 100_000
...
stoiip = ...
p90, p50, p10 = ...

line = ...
with open("kw3_summary.txt", "w", encoding="utf-8") as f:
    f.write(line + "\n")
print(line)

In [ ]:
grader.check("q5")

---
## The presentation (5 minutes, 4 slides)

Present to the "asset manager" as a team. One idea per slide, numbers with units:

1. **The data and its problems**: what the file contained, how many bad depths, what you did about them.
2. **The method**: the three pay rules, in plain words; why the low-resistivity sand is not pay.
3. **The result**: your `kw3_logs.png` figure, the pay zones and the net pay.
4. **The uncertainty and your recommendation**: P90 / P50 / P10, which parameter worries you most, and what you
   would measure next (a core? a test? a seismic survey?).

Be ready for one question about your code: any line of the notebook.

### Going further (not checked)

* Change the resistivity cut-off from 15 to 10 and to 25 Ω·m. How sensitive is the net pay? Show it in one small table.
* Move `evaluate` and `pay_zones` into your `geotools.py` from week 11 and import them.
* Which parameter widens the P10 − P90 range most? Fix each one at its most likely value in turn.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()